In [1]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD

# ── Arguments (Directly set for Colab) ──────────────────────────────────────────
save_model_flag   = -1  # Set to 1 to save the model after training
load_model_flag   = -1  # Set to 1 to load pre-trained weights
save_weights_path = "cnn_weights.weights.h5"

# ── Load MNIST dataset ─────────────────────────────────────────────────────────
print("Loading MNIST Dataset...")
dataset = fetch_openml("mnist_784", as_frame=False)

# Reshape to (samples, 28, 28, 1)  — channels LAST (TF default)
mnist_data   = dataset.data.reshape((-1, 28, 28, 1)).astype("float32") / 255.0
mnist_labels = dataset.target.astype("int")

train_img, test_img, train_labels, test_labels = train_test_split(
    mnist_data, mnist_labels, test_size=0.1, random_state=42
)

# One-hot encode labels
train_labels = to_categorical(train_labels, 10)
test_labels  = to_categorical(test_labels,  10)

# ── Build CNN model ────────────────────────────────────────────────────────────
def build_model(load_weights_path=None):
    model = keras.Sequential([
        # Conv block 1
        layers.Conv2D(20, (5, 5), padding="same", activation="relu",
                      input_shape=(28, 28, 1)),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Conv block 2
        layers.Conv2D(50, (5, 5), padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Conv block 3
        layers.Conv2D(100, (5, 5), padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Fully connected
        layers.Flatten(),
        layers.Dense(500, activation="relu"),

        # Output (10 classes: 0-9)
        layers.Dense(10, activation="softmax"),
    ])

    if load_weights_path and load_model_flag > 0:
        print(f"Loading saved weights from {load_weights_path} ...")
        model.load_weights(load_weights_path)

    return model

# ── Compile ────────────────────────────────────────────────────────────────────
print("\nCompiling model...")
sgd = SGD(learning_rate=0.01, decay=1e-6, momentum=0.9, nesterov=True)
clf = build_model(save_weights_path)
clf.compile(loss="categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])
clf.summary()

# ── Train ──────────────────────────────────────────────────────────────────────
if load_model_flag < 0:
    print("\nTraining the Model...")
    clf.fit(train_img, train_labels, batch_size=128, epochs=20, verbose=1)

    print("\nEvaluating accuracy...")
    loss, accuracy = clf.evaluate(test_img, test_labels, batch_size=128, verbose=1)
    print(f"\n✅ Model Accuracy: {accuracy * 100:.2f}%")

# ── Save weights ───────────────────────────────────────────────────────────────
if save_model_flag > 0:
    print(f"\nSaving weights to {save_weights_path} ...")
    clf.save_weights(save_weights_path)
    print("Weights saved!")

# ── Predict on random test samples ────────────────────────────────────────────
print("\n── Sample Predictions ──")
for num in np.random.choice(np.arange(0, len(test_labels)), size=(5,), replace=False):
    probs      = clf.predict(test_img[np.newaxis, num], verbose=0)
    prediction = probs.argmax(axis=1)[0]
    actual     = np.argmax(test_labels[num])
    result     = "✅" if prediction == actual else "❌"
    print(f"  {result}  Predicted: {prediction}   Actual: {actual}")

Loading MNIST Dataset...

Compiling model...


/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 20)     │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 20)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 50)     │        25,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 100)      │       125,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 100)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 900)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 500)            │       450,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 606,180 (2.31 MB)

 Trainable params: 606,180 (2.31 MB)

 Non-trainable params: 0 (0.00 B)


Training the Model...
Epoch 1/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.8986 - loss: 0.3322
Epoch 2/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9800 - loss: 0.0623
Epoch 3/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9864 - loss: 0.0440
Epoch 4/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9894 - loss: 0.0329
Epoch 5/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9920 - loss: 0.0259
Epoch 6/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9931 - loss: 0.0213
Epoch 7/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9945 - loss: 0.0172
Epoch 8/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9947 - loss: 0.0159
Epoch 9/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9966 - loss: 0.0113
Epoch 10/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9966 - loss: 0.0103
Epoch 11/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9975 - loss: 0.0084
Epoch 12/20
493/493 ━━━━━━━